# try/catch/finally → Python Exception Idioms

## What you'll practice

No checked exceptions, no `throws` clauses. Python adds `else` (runs only if no exception), prefers EAFP, chains causes with `from`,
groups failures with `ExceptionGroup`/`except*` (3.11+), and uses context managers instead of most `finally` blocks.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/java_to_python/test_04_05_exceptions.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/04_java_to_python/05_exceptions.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · try / except / else / finally

**Implement:** `load_json_config(path) -> dict`

Return the parsed JSON object from `path`. Missing file → return `{}`. Invalid JSON → raise `ValueError` whose message contains the
file name and whose `__cause__` is the `JSONDecodeError`. A valid JSON that isn't an object (e.g. a list) → `ValueError`. Use `try/except/else`.

**Input:** `path`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_load_json_config` in `tests/java_to_python/test_04_05_exceptions.py`.

**Check:** `pytest tests/java_to_python/test_04_05_exceptions.py -k test_load_json_config`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Fallback on transient errors only

**Implement:** `fetch_with_fallback(primary, fallback) -> tuple[object, str]`

Call `primary()`; on `ConnectionError` or `TimeoutError` call `fallback()` and return `(value, "fallback")`, otherwise `(value, "primary")`.
Any other exception from `primary` (e.g. `KeyError`, a bug) must propagate unchanged. If the fallback also fails, raise its exception — but
make sure the original primary error is attached (Python does this automatically as `__context__` when raising inside `except`).

**Input:** `primary, fallback`

**Expected output:** `tuple[object, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_fetch_with_fallback, test_fetch_with_fallback_both_fail_keeps_context` in `tests/java_to_python/test_04_05_exceptions.py`.

**Check:** `pytest tests/java_to_python/test_04_05_exceptions.py -k "test_fetch_with_fallback or test_fetch_with_fallback_both_fail_keeps_context"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Report all failures with ExceptionGroup

**Implement:** `validate_or_raise(records: list[dict], validator) -> list[dict]`

Run `validator(record)` (which raises `ValueError` for bad records) on every record. If any fail, raise
`ExceptionGroup("N invalid records", [errors...])` containing **all** the `ValueError`s, each annotated with a note
(`err.add_note(f"record index {i}")`). Other exception types must propagate immediately. Return the records unchanged if all pass.
Callers can then use `except* ValueError as eg:` (Python 3.11+).

**Input:** `records: list[dict], validator`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_validate_or_raise_ok, test_validate_or_raise_group, test_validate_or_raise_other_errors_propagate` in `tests/java_to_python/test_04_05_exceptions.py`.

**Check:** `pytest tests/java_to_python/test_04_05_exceptions.py -k "test_validate_or_raise_ok or test_validate_or_raise_group or test_validate_or_raise_other_errors_propagate or test_except_star_usage"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Translate low-level errors at a boundary

**Implement:** `class StorageError(Exception)`, `load_records(path) -> list[dict]`

A storage layer must not leak `OSError`/`UnicodeDecodeError`/`JSONDecodeError` to the pipeline. `load_records(path)` reads a JSON array of
objects. Translate: missing file → `StorageError("not found: <path>")`; permission or other `OSError` → `StorageError("cannot read: <path>")`;
undecodable bytes → `StorageError("encoding error: <path>")`; invalid JSON or a non-list/non-dict-items payload → `StorageError("corrupt: <path>")`.
Always chain with `from` so the root cause stays visible. `StorageError` exposes `.path`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_load_records_ok, test_load_records_errors, test_load_records_directory_is_oserror` in `tests/java_to_python/test_04_05_exceptions.py`.

**Check:** `pytest tests/java_to_python/test_04_05_exceptions.py -k "test_load_records_ok or test_load_records_errors or test_load_records_directory_is_oserror"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Atomic file writes with guaranteed cleanup

**Implement:** `write_json_atomic(path, payload) -> None`

Readers must never see a half-written file. Write `payload` as JSON to a temporary file **in the same directory** (`tempfile.NamedTemporaryFile(dir=..., delete=False)`
or `mkstemp`), `flush` + `os.fsync`, then `os.replace(tmp, path)` (atomic on POSIX and Windows). If serialization or writing fails, delete the temp
file and re-raise the original exception; the existing target file must remain untouched and no `*.tmp` leftovers may remain. Create parent dirs if needed.

**Input:** `path, payload`

**Expected output:** `None`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_write_json_atomic, test_write_json_atomic_failure_keeps_original` in `tests/java_to_python/test_04_05_exceptions.py`.

**Check:** `pytest tests/java_to_python/test_04_05_exceptions.py -k "test_write_json_atomic or test_write_json_atomic_failure_keeps_original"`

**YOUR CODE ↓**